# Модуль 04 · Урок 4. Сводные таблицы, melt, crosstab, MultiIndex

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Широкие и длинные таблицы
Длинная таблица хранит измерения в отдельных столбцах: `month`, `department`, `sales`. Широкая размещает значения измерения в заголовках: `sales_Jan`, `sales_Feb`. Для анализа и группировок обычно удобна длинная форма, для отчёта — широкая. `pivot()` меняет форму без агрегации и выдаёт ошибку при дублирующихся сочетаниях ключей; `pivot_table()` выполняет агрегацию (`aggfunc`).

In [ ]:
import pandas as pd
scores = pd.DataFrame({'person':['A','A','B','B'],'subject':['math','art','math','art'],'points':[80,90,75,82]})
print(scores.pivot(index='person',columns='subject',values='points'))

## 2. pivot_table и пропуски
При одинаковых сочетаниях индекс/колонка `pivot_table(...,aggfunc='sum')` суммирует значения. Параметр `fill_value=0` заполняет отсутствующие **комбинации** в результате, но 0 может быть семантически неверным, если отсутствие данных означает неизвестность. Отдельно решайте, нужны ли итоги (`margins=True`).

## 3. melt и crosstab
`melt(id_vars=...,value_vars=...,var_name=...,value_name=...)` собирает несколько колонок в пары «название измерения / значение». `crosstab()` строит таблицу частот для двух категориальных признаков; при `normalize='index'` получает доли внутри строк. Эти операции помогают строить матрицы переходов и анализировать категориальные распределения.

In [ ]:
wide = pd.DataFrame({'city':['A','B'],'jan':[10,20],'feb':[15,25]})
print(wide.melt(id_vars='city',var_name='month',value_name='sales'))

## 4. Многоуровневый индекс
Группировка по двум ключам часто создаёт MultiIndex: уровень `city` и уровень `status`. К отдельному уровню можно обращаться `xs`, а `unstack()` превращает уровень индекса в столбцы. Если MultiIndex мешает, `reset_index()` возвращает плоскую таблицу. Пользуйтесь сложным индексом тогда, когда он делает анализ проще, а не ради самого использования.

**Проверьте себя:** когда `pivot` выдаёт ошибку, а `pivot_table` работает? Для чего нужен `melt`? Что означают уровни MultiIndex?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m4_16 · pivot status

Получите сводную таблицу: индекс `city`, столбцы — `status`, значения — количество заказов (size), недостающие сочетания заполните 0.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m4_16: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m4_16 = expected_for("m4_16", df=df)


In [ ]:
# m4_16: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 3 столбца; колонки: `cancelled`, `new`, `paid`.

```text
status  cancelled  new  paid
city                        
Казань          0    1     1
Москва          0    0     2
Тверь           1    1     0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m4_16", result, _expected_m4_16)

### m4_17 · city category matrix

Постройте таблицу частот `pd.crosstab` город × категория, отсортированную по стандартным правилам pandas.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m4_17: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m4_17 = expected_for("m4_17", df=df)


In [ ]:
# m4_17: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 3 столбца; колонки: `книги`, `кофе`, `чай`.

```text
category  книги  кофе  чай
city                      
Казань        1     0    1
Москва        0     0    2
Тверь         0     2    0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m4_17", result, _expected_m4_17)

### m4_18 · long prices

Для каждой строки оставьте `order_id`, а `price` и `qty` разполучите в длинную таблицу с колонками `order_id`,`metric`,`value` (порядок: сначала price для всех строк, затем qty).

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m4_18: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m4_18 = expected_for("m4_18", df=df)


In [ ]:
# m4_18: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 12 строк, 3 столбца; колонки: `order_id`, `metric`, `value`.

```text
  order_id metric  value
0  demo_o1  price  127.0
1  demo_o2  price  247.0
2  demo_o3  price   97.0
3  demo_o4  price  317.0
4  demo_o5  price   57.0
5  demo_o6  price  147.0
... (показаны только первые строки)
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m4_18", result, _expected_m4_18)

### m4_19 · multiindex sales

Получите Series с суммой `price * qty` по `(city,category)`, используя MultiIndex и стандартную сортировку ключей.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m4_19: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m4_19 = expected_for("m4_19", df=df)


In [ ]:
# m4_19: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 4 значений; имя серии: `revenue`.

```text
city    category
Казань  книги       317.0
        чай           0.0
Москва  чай         642.0
Тверь   кофе        418.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m4_19", result, _expected_m4_19)

### m4_20 · unstack status

Посчитайте число заказов по `(city,status)` через groupby.size, затем разполучите status в столбцы и заполните отсутствующие сочетания нулями.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m4_20: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m4_20 = expected_for("m4_20", df=df)


In [ ]:
# m4_20: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 3 столбца; колонки: `cancelled`, `new`, `paid`.

```text
status  cancelled  new  paid
city                        
Казань          0    1     1
Москва          0    0     2
Тверь           1    1     0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m4_20", result, _expected_m4_20)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()